# Using YOLOv8 to detect grocery products

This notebook follows the same steps as the blood cell notebook, but uses real photos of grocery store shelves.
It finds 20 products, plus `other_product` for everything else on the shelf, and then looks up their prices.

Dataset: NorgesGruppen Grocery Detection Dataset (CC BY-NC 4.0), https://huggingface.co/datasets/valiantlynxz/norwegian-grocery

In [ ]:
'''
The ultralytics Python library is the official way
to install and use YOLOv8
'''

%pip install ultralytics huggingface_hub

In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import shutil
from shutil import copyfile
import matplotlib.pyplot as plt
from matplotlib import animation, rc
rc('animation', html='jshtml')
from PIL import Image
import ultralytics
from ultralytics import YOLO
ultralytics.checks()

In [ ]:
# Get the current working directory
base_dir = os.getcwd()

# Construct dataset paths relative to current folder
dataset_dir = os.path.join(base_dir, "dataset/grocery_dataset")
train_path = os.path.join(dataset_dir, "train/images")
valid_path = os.path.join(dataset_dir, "valid/images")
test_path  = os.path.join(dataset_dir, "test/images")
test_label_path = os.path.join(dataset_dir, "test/labels")   # YOLO label .txt files live here
done_file = os.path.join(dataset_dir, "prepared.txt")
runs_dir = os.path.join(base_dir, "runs", "detect")   # keep all results inside this project

# Raw download: looked for in this folder, then in the folder above
raw_folders = [os.path.join(base_dir, "NorwegianGrocery"), os.path.join(base_dir, "..", "NorwegianGrocery")]
raw_dir = next((p for p in raw_folders if os.path.exists(os.path.join(p, "train", "annotations.json"))), raw_folders[0])


# Prepare the dataset

The raw dataset has 356 products and only 248 photos, which is too little to learn every product.
So this notebook keeps the 5 most common products from each of the 4 store sections. The other products stay in the labels as `other_product`.

In [ ]:
# The 20 products this project detects: the 5 most common products in each store section.
# Every other product on the shelf is kept as one extra class, 'other_product', so the
# model is never taught that a real product is "background".
PRODUCTS = {
    'crispbread': ['HAVRE KNEKKEBRØD 300G WASA',
                   'KNEKKEBRØD 100 FRØ&HAVSALT 245G WASA',
                   'KNEKKEBRØD SPORT+ 210G WASA',
                   'HUSMAN KNEKKEBRØD 260G WASA',
                   'FIBER BALANCE 230G WASA'],
    'cereal':     ['CHEERIOS HAVRE 375G NESTLE',
                   'NESQUIK FROKOSTBLANDING 375G NESTLE',
                   'CHEERIOS MULTI 375G NESTLE',
                   "SMACKS 330G KELLOGG'S",
                   'SUPERGRØT KANEL&PUFFET QUINOA 54G'],
    'hot_drinks': ['EVERGOOD CLASSIC FILTERMALT 250G',
                   'ALI ORIGINAL FILTERMALT 250G',
                   'EVERGOOD CLASSIC KOKMALT 250G',
                   "O'BOY ORIGINAL 450G FREIA",
                   'GRØNN TE SITRON 25POS TWININGS'],
    'eggs':       ['FROKOSTEGG FRITTGÅENDE L 12STK PRIOR',
                   'EGG FRITTGÅENDE 18STK S/M FIRST PRICE',
                   'EGG ØKOLOGISK M/L/XL 6STK PRIOR',
                   'EGG FRITTGÅENDE M/L/XL SOLEGG 6STK PRIOR',
                   'GÅRDSEGG M/L 12STK ELDORADO'],
}
class_names = [name for section in PRODUCTS.values() for name in section] + ['other_product']
OTHER_ID = len(class_names) - 1
print(len(class_names), 'classes')

In [ ]:
# Download the raw dataset (about 0.9 GB) only if it is not on this computer yet
if not os.path.exists(done_file) and not os.path.exists(os.path.join(raw_dir, 'train', 'annotations.json')):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='valiantlynxz/norwegian-grocery', repo_type='dataset',
                      local_dir=raw_dir, allow_patterns=['train/*'])
print('Raw dataset folder:', raw_dir)

In [ ]:
# Convert the raw COCO annotations into the same folder layout as the blood cell dataset:
#   dataset/grocery_dataset/{train,valid,test}/{images,labels}
# COCO box = [x, y, width, height] in pixels
# YOLO box = class x_center y_center width height, all divided by the image size (0 to 1)
import json
from collections import Counter, defaultdict

SEED = 42        # same split every time
MAX_SIDE = 1280  # photos are up to 5712 px wide; smaller copies make training much faster

def prepare_dataset():
    with open(os.path.join(raw_dir, 'train', 'annotations.json'), encoding='utf-8') as f:
        coco = json.load(f)

    # raw category id -> our class id
    name_to_class = {name: i for i, name in enumerate(class_names)}
    id_to_class = {c['id']: name_to_class.get(c['name'], OTHER_ID) for c in coco['categories']}
    missing = set(class_names[:-1]) - {c['name'] for c in coco['categories']}
    assert not missing, f'not found in annotations.json: {missing}'

    boxes = defaultdict(list)
    for a in coco['annotations']:
        boxes[a['image_id']].append(a)

    # Each image belongs to the store section whose products it shows the most
    section_of_class = {name_to_class[n]: s for s, names in PRODUCTS.items() for n in names}
    sections = defaultdict(list)
    for image in coco['images']:
        seen = Counter(section_of_class.get(id_to_class[a['category_id']]) for a in boxes[image['id']])
        seen.pop(None, None)
        sections[seen.most_common(1)[0][0] if seen else 'other'].append(image)

    # 70% train / 15% valid / 15% test inside every section, so each split has every section
    splits = {'train': [], 'valid': [], 'test': []}
    rng = random.Random(SEED)
    for section in sorted(sections):
        images = sorted(sections[section], key=lambda image: image['file_name'])
        rng.shuffle(images)
        n_valid = n_test = round(len(images) * 0.15)
        splits['valid'] += images[:n_valid]
        splits['test']  += images[n_valid:n_valid + n_test]
        splits['train'] += images[n_valid + n_test:]

    for split, images in splits.items():
        os.makedirs(os.path.join(dataset_dir, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(dataset_dir, split, 'labels'), exist_ok=True)
        for image in tqdm(images, desc=split):
            W, H = image['width'], image['height']
            stem = os.path.splitext(image['file_name'])[0]

            # Every image becomes .jpg because some raw files are .jpeg
            with Image.open(os.path.join(raw_dir, 'train', 'images', image['file_name'])) as im:
                im = im.convert('RGB')
                im.thumbnail((MAX_SIDE, MAX_SIDE))
                im.save(os.path.join(dataset_dir, split, 'images', stem + '.jpg'), quality=90)

            with open(os.path.join(dataset_dir, split, 'labels', stem + '.txt'), 'w') as f:
                for a in boxes[image['id']]:
                    x, y, w, h = a['bbox']
                    x1, y1 = max(x, 0), max(y, 0)
                    x2, y2 = min(x + w, W), min(y + h, H)
                    f.write(f"{id_to_class[a['category_id']]} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} "
                            f"{(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}\n")
    counts = {split: len(images) for split, images in splits.items()}
    with open(done_file, 'w') as f:   # written last, so a stopped run is simply redone
        f.write(str(counts))
    return counts

if os.path.exists(done_file):
    print('Dataset is already prepared:', dataset_dir)
else:
    print(prepare_dataset())

Check the labels before training: the boxes should sit on the products. Colored boxes are the 20 products and grey boxes are `other_product`.

In [ ]:
# Draw the labels of the first training image
sample = sorted(os.listdir(train_path))[0]
image = cv2.imread(os.path.join(train_path, sample))
H, W = image.shape[0:2]
with open(os.path.join(dataset_dir, 'train/labels', os.path.splitext(sample)[0] + '.txt')) as f:
    for line in f:
        c, xc, yc, w, h = line.split()
        c, xc, yc, w, h = int(c), float(xc), float(yc), float(w), float(h)
        p1 = (int((xc - w / 2) * W), int((yc - h / 2) * H))
        p2 = (int((xc + w / 2) * W), int((yc + h / 2) * H))
        cv2.rectangle(image, p1, p2, (170, 170, 170) if c == OTHER_ID else (0, 255, 0), 2)
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
# Define Bounding Box Color
# One fixed color per class (the blood cell notebook only needed 3)
rng = random.Random(0)
colors=[tuple(rng.randint(60,255) for _ in range(3)) for _ in class_names]
colors[OTHER_ID]=(170,170,170)   # other_product is grey

# Create yaml file 

In [ ]:
# Our custom dataset info

import yaml

data_yaml = dict(
    train = train_path,
    val   = valid_path,
    test  = test_path,
    nc    = len(class_names),
    names = class_names   # same list used to write the labels, so the IDs always match
)

with open('data.yaml', 'w', encoding='utf-8') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=False, allow_unicode=True)

# show the file (was "!cat data.yaml." - the trailing dot made the filename wrong)
print(open('data.yaml', encoding='utf-8').read())


# Train

In [ ]:
# Load a pretrained YOLOv8 model
# yolov8x (68M params) is VERY slow on a laptop CPU. yolov8n / yolov8s train in minutes.
# Switch back to "yolov8x.pt" if you have time or a GPU.
model = YOLO("yolov8n.pt")


In [ ]:
# Train YOLO model with custom dataset
import torch

# Use a GPU if there is one: NVIDIA (Windows/Linux) or Apple (Mac). Otherwise use the CPU.
if torch.cuda.is_available():
    device = 0
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Training on:", device)

EPOCHS  = 80
IMGSZ   = 960   # shelf products are small, so 480 is too low
BATCH   = 8     # lower this to 4 if you get an out of memory error
WORKERS = 0 if os.name == "nt" else 4   # Windows notebooks are safest with 0

model.train(data="data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=device, workers=WORKERS, project=runs_dir)

# YOLO saves each new run to runs/detect/train, train2, train3, ...
# so take the real folder from the trainer instead of hard-coding it
train_dir = str(model.trainer.save_dir)
print("Training results saved to:", train_dir)


# Result of Training

In [ ]:
paths2=[]
for dirname, _, filenames in os.walk(train_dir):   # was '/runs/detect/train' (leading / = root of disk)
    for filename in filenames:
        if filename[-4:]=='.jpg':
            paths2+=[(os.path.join(dirname, filename))]
paths2=sorted(paths2)


In [ ]:
for path in paths2:
    image = Image.open(path)
    image=np.array(image)
    plt.figure(figsize=(20,10))
    plt.imshow(image)
    plt.show()

In [ ]:
# Loss curves, mAP curves and the confusion matrix (these are .png, so the cell above skips them)
for name in ['results.png', 'confusion_matrix_normalized.png']:
    path = os.path.join(train_dir, name)
    if os.path.exists(path):
        plt.figure(figsize=(16, 10))
        plt.imshow(np.array(Image.open(path)))
        plt.axis('off')
        plt.show()

# Predict

In [ ]:
best_path0 = os.path.join(train_dir, 'weights', 'best.pt')
source0 = test_path   # test_path already ends in /images (old code made .../images/images)

# If you restarted the kernel and don't want to retrain, set the run folder by hand, e.g.:
# best_path0 = os.path.join(base_dir, 'runs/detect/train/weights/best.pt')
print(best_path0, os.path.exists(best_path0))


In [ ]:
#test images
ppaths=[]
for dirname, _, filenames in os.walk(source0):
    for filename in filenames:
        if filename[-4:]=='.jpg':
            ppaths+=[(os.path.join(dirname, filename))]
ppaths=sorted(ppaths)
#print(ppaths[0])
#print(len(ppaths))

In [ ]:
model2 = YOLO(best_path0)

In [ ]:
# CLI version (optional) - quotes are required because the path contains a space
!yolo task=detect mode=predict model="{best_path0}" conf=0.2 imgsz={IMGSZ} project="{runs_dir}" source="{source0}"


# Result of Prediction

In [ ]:
results = model2.predict(source0,conf=0.2,imgsz=IMGSZ,device=device,verbose=False)
print(len(results))

PBOX for test predicted

In [ ]:
PBOX=pd.DataFrame(columns=range(6))
for i in range(len(results)):#len(results)
    arri=pd.DataFrame(results[i].boxes.data.cpu().numpy()).astype(float)
    path=ppaths[i]
    file=os.path.basename(path)   # works on Windows too
    arri=arri.assign(file=file)
    arri=arri.assign(i=i)
    PBOX=pd.concat([PBOX,arri],axis=0)
    
PBOX.columns=['x1','y1','x2','y2','confidence','class','file','i']
PBOX['class']=PBOX['class'].astype(int) #.apply(lambda x: class_map[int(x)])
PBOX['i']=PBOX['i'].astype(int)

PBOX=PBOX.reset_index(drop=True)
display(PBOX)

In [ ]:
def draw_box2(n0,ppaths,PBOX):
    
    ipath=ppaths[n0]
    image=cv2.imread(ipath)
    ipath0=ppaths[0]
    image0=cv2.imread(ipath0)
    W,H=image0[0:2]
    file=os.path.basename(ipath)   # works on Windows too
    
    if PBOX[PBOX['file']==file] is not None:
        box=PBOX[PBOX['file']==file]
        box=box.reset_index(drop=True)
        #display(box)

        for i in range(len(box)):
            #conf=box.loc[i,'confidence']
            label=int(box.loc[i,'class'])
            x1=int(box.loc[i,'x1'])
            y1=int(box.loc[i,'y1'])
            x2=int(box.loc[i,'x2']) 
            y2=int(box.loc[i,'y2'])
            #print(label,x,y,x2,y2)
            
            cv2.putText(image, f'{label}', (x1, y1-5), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, colors[label],2)
            
            cv2.rectangle(image,(x1,y1),(x2,y2),colors[label],1) #green
    
    #plt.imshow(image)
    #plt.show()   
    
    return image

In [ ]:
def create_animation(ims):
    
    fig=plt.figure(figsize=(12,8))
    im=plt.imshow(cv2.cvtColor(ims[0],cv2.COLOR_BGR2RGB))
    text = plt.text(0.05, 0.05, f'Slide {0}', transform=fig.transFigure, fontsize=14, color='blue')
    plt.axis('off')
    plt.close()

    def animate_func(i):
        im.set_array(cv2.cvtColor(ims[i],cv2.COLOR_BGR2RGB))
        text.set_text(f'Slide {i}')        
        return [im]    
    
    return animation.FuncAnimation(fig, animate_func, frames=len(ims), interval=1000)

test images with prediced rectangles

In [ ]:
images2=[]
for i in tqdm(range(len(ppaths))):
    images2+=[draw_box2(i,ppaths,PBOX)]
    
#print('Test Predicted')
#create_animation(images2)

# Process Test Annotation Data for Test True Data

In [ ]:
impaths=[]
txtpaths=[]
for dirname, _, filenames in os.walk(test_path):        # images
    for filename in filenames:
        if filename.endswith('.jpg'):
            impaths+=[os.path.join(dirname, filename)]
for dirname, _, filenames in os.walk(test_label_path):  # labels are in test/labels, not test/images
    for filename in filenames:
        if filename.endswith('.txt'):
            txtpaths+=[os.path.join(dirname, filename)]
impaths.sort()
txtpaths.sort()
print(len(impaths), 'images,', len(txtpaths), 'label files')


In [ ]:
dirimg = test_path   # was a leftover Kaggle path ('/kaggle/input/brain-tumor/...')
boxdata=[]
for i in range(len(txtpaths)):
    path=txtpaths[i]
    filename=os.path.splitext(os.path.basename(path))[0]
    ipath=os.path.join(dirimg,filename+'.jpg')
    image=cv2.imread(ipath)
    if image is None:
        print('image not found for', path)
        continue
    H,W=image.shape[0:2]   # shape is (height, width, channels)
    file=os.path.basename(ipath)

    with open(path) as f:
        lines=[l.split() for l in f if l.strip()]
    for parts in lines:
        boxdata+=[parts[:5]+[W,H,file,i]]
print(len(boxdata), 'ground-truth boxes')


BOX for test true

In [ ]:
BOX=pd.DataFrame(data=boxdata,columns=['class','x_center','y_center','width','height','W','H','file','i'])
display(BOX)

In [ ]:
BOX['class']=BOX['class'].astype(int)
BOX['x_center']=BOX['x_center'].astype(float)
BOX['y_center']=BOX['y_center'].astype(float)
BOX['width']=BOX['width'].astype(float)
BOX['height']=BOX['height'].astype(float)
BOX['W']=BOX['W'].astype(int)
BOX['H']=BOX['H'].astype(int)
BOX['i']=BOX['i'].astype(int)

# Convert the coordinates in `BOX` from relative to absolute
BOX['x1'] = (BOX['x_center'] - BOX['width'] / 2) * BOX['W']
BOX['y1'] = (BOX['y_center'] - BOX['height'] / 2) * BOX['H']
BOX['x2'] = (BOX['x_center'] + BOX['width'] / 2) * BOX['W']
BOX['y2'] = (BOX['y_center'] + BOX['height'] / 2) * BOX['H']

display(BOX)

test images with true rectangles

In [ ]:
tpaths2=[]
for dirname, _, filenames in os.walk(source0):
    for filename in filenames:
        if filename[-4:]=='.jpg':
            tpaths2+=[(os.path.join(dirname, filename))]
tpaths2=sorted(tpaths2)

timages2=[]
for i in tqdm(range(len(tpaths2))):
    timages2+=[draw_box2(i,tpaths2,BOX)]


# Comparison between true and predicted rectangles in test data

In [ ]:
print('Test True')
SHOW = 6   # first 6 test images only, so the notebook file stays small
create_animation(timages2[:SHOW])

In [ ]:
print('Test Predicted')
create_animation(images2[:SHOW])

# Mean Average Precision (mAP)

This is the most common metric for evaluating object detection models. mAP calculates the average precision across all classes. It measures how well the model's predicted bounding boxes match the ground truth bounding boxes, considering both the precision (how many of the predicted boxes are correct) and recall (how many correct boxes the model detects).
mAP is calculated over a range of IoU (Intersection over Union) thresholds, such as 0.5 (mAP@0.5).

In [ ]:
# IoU calculation function
def calculate_iou(pred_bbox, true_bbox):
    x1_pred, y1_pred, x2_pred, y2_pred = pred_bbox
    x1_true, y1_true, x2_true, y2_true = true_bbox
    
    # Calculate intersection area
    x1_inter = max(x1_pred, x1_true)
    y1_inter = max(y1_pred, y1_true)
    x2_inter = min(x2_pred, x2_true)
    y2_inter = min(y2_pred, y2_true)
    
    # Calculate areas
    inter_area = max(0, x2_inter - x1_inter) * max(0, y2_inter - y1_inter)
    pred_area = (x2_pred - x1_pred) * (y2_pred - y1_pred)
    true_area = (x2_true - x1_true) * (y2_true - y1_true)
    
    # IoU
    union_area = pred_area + true_area - inter_area
    iou = inter_area / union_area
    
    return iou,union_area, inter_area, pred_area, true_area

In [ ]:
# Matching predicted results with true labels based on image names
# Each shelf photo has about 90 boxes, so every prediction is compared with all true boxes of
# the same class. Each true box can only be matched once, and the most confident predictions go first.
matched_results = []
matched_true = 0

for file, preds in PBOX.groupby('file'):
    true_data = BOX[BOX['file'] == file]
    used = set()
    for _, pred_row in preds.sort_values('confidence', ascending=False).iterrows():
        pred_bbox = [pred_row['x1'], pred_row['y1'], pred_row['x2'], pred_row['y2']]
        best = (0, 0, 0, 0, 0)   # iou, union_area, inter_area, pred_area, true_area
        best_j = None
        for j, true_row in true_data[true_data['class'] == pred_row['class']].iterrows():
            if j in used:
                continue
            true_bbox = [true_row['x1'], true_row['y1'], true_row['x2'], true_row['y2']]
            result = calculate_iou(pred_bbox, true_bbox)
            if result[0] > best[0]:
                best, best_j = result, j

        iou, union_area, inter_area, pred_area, true_area = best
        tp = iou > 0.5   # If IoU exceeds the threshold, it's a True Positive
        if tp:
            used.add(best_j)
        matched_results.append({'image': pred_row['file'], 'iou': iou,'union_area':union_area, 'inter_area':inter_area, 'pred_area':pred_area, 'true_area':true_area, 'tp': tp})
    matched_true += len(used)
# To calculate mAP, you would typically aggregate the results based on different confidence thresholds
# and calculate Precision-Recall curve for each class to get the AP.
# Here, we use a simple threshold of IoU > 0.5 for TP and FP counts.


    image: The image file name (e.g. img_00001.jpg).
    iou: The IoU value with the corresponding Ground Truth box (range 0.0 to 1.0).
    tp: Indicates whether the prediction is True Positive (True) (FP if False).

In [ ]:
# Create a DataFrame for the matched results
matched_df = pd.DataFrame(matched_results)
display(matched_df)

In [ ]:
import matplotlib.pyplot as plt

# Check for NaNs
nan_values = matched_df['iou'].isna().sum()
print(f"Number of NaN values: {nan_values}")

# Check for infinite values
inf_values = np.isinf(matched_df['iou']).sum()
print(f"Number of infinite values: {inf_values}")

valid_iou_values = matched_df['iou'][np.isfinite(matched_df['iou'])]

plt.hist(valid_iou_values, bins=50)
plt.title("IoU Distribution")
plt.xlabel("IoU")
plt.xlim(0,1) 
plt.show()

In [ ]:
# Simple scores at conf=0.2 and IoU > 0.5
tp = int(matched_df['tp'].sum())
fp = len(matched_df) - tp
fn = len(BOX) - matched_true
print(f"True positives : {tp}")
print(f"False positives: {fp}")
print(f"Missed boxes   : {fn}")
print(f"Precision      : {tp / (tp + fp):.3f}")
print(f"Recall         : {tp / (tp + fn):.3f}")

In [ ]:
# Real mAP computed by Ultralytics on the test split (handles multiple boxes & classes correctly)
metrics = model2.val(data="data.yaml", split="test", imgsz=IMGSZ, batch=BATCH, device=device, workers=WORKERS, project=runs_dir)
print("mAP@0.5     :", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)


In [ ]:
# Score for each class on the test split
per_class = pd.DataFrame({
    'class': [int(c) for c in metrics.box.ap_class_index],
    'product': [class_names[int(c)] for c in metrics.box.ap_class_index],
    'mAP@0.5': metrics.box.ap50.round(3),
    'mAP@0.5:0.95': metrics.box.ap.round(3),
}).sort_values('mAP@0.5', ascending=False).reset_index(drop=True)
display(per_class)

# Price lookup

YOLO only finds the products. The price comes from `prices.csv`, which is our own table.
**The prices in that file are examples, not real store prices.** Change them if you want real prices.

In [ ]:
# Product -> price
prices = pd.read_csv('prices.csv', encoding='utf-8')
price_of = dict(zip(prices['product'], prices['price_nok']))
assert set(price_of) == set(class_names[:-1]), 'prices.csv must list the same 20 products as class_names'
display(prices)

In [ ]:
PRICE_CONF = 0.5   # only count confident detections

def price_report(file):
    pred = PBOX[(PBOX['file'] == file) & (PBOX['confidence'] >= PRICE_CONF) & (PBOX['class'] != OTHER_ID)]
    true = BOX[(BOX['file'] == file) & (BOX['class'] != OTHER_ID)]
    rows = []
    for c in sorted(set(pred['class']) | set(true['class'])):
        name = class_names[c]
        detected = int((pred['class'] == c).sum())
        rows.append({'class': c, 'product': name, 'detected': detected,
                     'true count': int((true['class'] == c).sum()),
                     'price (NOK)': price_of[name], 'subtotal (NOK)': round(detected * price_of[name], 2)})
    return pd.DataFrame(rows), pred

# Use the test photo where the model found the most priced products
counts = PBOX[(PBOX['confidence'] >= PRICE_CONF) & (PBOX['class'] != OTHER_ID)]['file'].value_counts()
if counts.empty:
    print(f'No product reached confidence {PRICE_CONF}. Train for more epochs or lower PRICE_CONF.')
else:
    demo_file = counts.index[0]
    report, demo_boxes = price_report(demo_file)
    print(demo_file)
    display(report)
    print(f"Value of the detected products: {report['subtotal (NOK)'].sum():.2f} NOK")
    print(f"Items detected: {report['detected'].sum()} of {report['true count'].sum()} in the true labels")

    demo_index = [os.path.basename(p) for p in ppaths].index(demo_file)
    plt.figure(figsize=(14, 10))
    plt.imshow(cv2.cvtColor(draw_box2(demo_index, ppaths, demo_boxes), cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

In [ ]:
# Try your own photo: put the file path here, for example 'my_shelf.jpg'
my_photo = None

if my_photo:
    found = model2.predict(my_photo, conf=PRICE_CONF, imgsz=IMGSZ, device=device, verbose=False)[0]
    names_found = [class_names[int(c)] for c in found.boxes.cls]
    for name in sorted(set(names_found)):
        count = names_found.count(name)
        print(f"{count} x {name}", f"= {count * price_of[name]:.2f} NOK" if name in price_of else "(no price)")
    plt.figure(figsize=(14, 10))
    plt.imshow(found.plot()[:, :, ::-1])
    plt.axis('off')
    plt.show()

# Results and limits

Use your own numbers from this run in the report:

- **Main score:** `mAP@0.5` and `mAP@0.5:0.95` from the test split.
- **Best and worst products:** the per-class table.
- **Price result:** the price report shows detected and true counts side by side.

**Limits:**
- The model only knows 20 products. Everything else is `other_product`.
- The dataset has only 248 photos, so the test split is small (37 photos).
- Some test photos may come from the same stores as training photos.
- `prices.csv` contains example prices, not real store prices.